# Práctica 1: Ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [scikit-learn](https://scikit-learn.org) de Python.

### Ejercicio 1

El fichero `cars.csv` contiene información acerca de la idoneidad de una serie de coches, en función de los siguientes atributos discretos:

* Precio de compra (`buying`): posibles valores `vhigh`, `high`, `med`, `low`.
* Coste de mantenimiento (`maint`): posibles valores `vhigh`, `high`, `med`, `low`.
* Número de puertas (`doors`): posibles valores `2`, `3`, `4`, `5more`.
* Número de asientos (`persons`): posibles valores `2`, `4`, `more`.
* Tamaño del maletero (`lug_boot`): posibles valores `small`, `med`, `big`.
* Nivel de seguridad estimada (`safety`): posibles valores `low`, `med`, `high`.

La idoneidad de cada coche se indica mediante el atributo `acceptability`, que los clasifica como `unacc`, `acc`, `good` o `vgood`.

Se pide realizar lo siguiente:

1. Estimar mediante validación cruzada la tasa de acierto que obtendría un modelo naive Bayes para distintos valores del parámetro de suavizado.
2. Seleccionar el mejor valor de suavizado, entrenar un modelo naive Bayes a partir de todos los ejemplos usados para la validación cruzada y proporcionar su tasa de acierto sobre un conjunto de prueba reservado desde el principio.

In [38]:
import pandas as pd

cars = pd.read_csv('cars.csv')

categorias = ['buying', 'maint', 'doors', 'persons', 'lug_boot', 'safety']
clase = ['acceptability']
atributos = cars.loc[:,categorias]
objetivo = cars[clase]

# Convertimos los valores discretos a números
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder

oe = OrdinalEncoder()
atributos = oe.fit_transform(atributos)
le = LabelEncoder()
objetivo = le.fit_transform(objetivo)

# Separamos atributos de entrenamiento y pruebas
from sklearn.model_selection import train_test_split

(atributos_entrenamiento, atributos_prueba,
 objetivo_entrenamiento, objetivo_prueba) = train_test_split(atributos, objetivo, test_size=.2, stratify=objetivo)

# Clasificador Naive Bayes
from sklearn.naive_bayes import CategoricalNB
from sklearn.model_selection import GridSearchCV
from inspect import signature
from sklearn.metrics import accuracy_score

# Extraer los parámetros
print(signature(CategoricalNB))

# Rejilla
rejilla_de_hiperparametros = { 'alpha': range(1,6), 'force_alpha': [True, False]
                              }
# Búsqueda en Rejilla

busqueda_en_rejilla = GridSearchCV(CategoricalNB(),
                                   rejilla_de_hiperparametros,
                                   cv=10,
                                   scoring='accuracy'
                                   #scoring=['accuracy', 'precision_macro', 'recall_macro', 'f1_macro']
                                   )

busqueda_en_rejilla.fit(atributos_entrenamiento,objetivo_entrenamiento)
mejor_modelo = busqueda_en_rejilla.best_estimator_
mejor_modelo.score(atributos_prueba,objetivo_prueba)

busqueda_en_rejilla.cv_results_

busqueda_en_rejilla.best_params_





/home/juanan/ETSII_IA/jupyter-env/lib/python3.12/site-packages/sklearn/preprocessing/_label.py:120: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(*, alpha=1.0, force_alpha=True, fit_prior=True, class_prior=None, min_categories=None)


{'alpha': 1, 'force_alpha': True}

### Ejercicio 2

Los púlsares son un tipo raro de estrella de neutrones que produce emisiones de radio detectables aquí en la Tierra. Son de considerable interés científico como sondas del espacio-tiempo, el medio interestelar y los estados de la materia.

A medida que los púlsares giran, su haz de emisión recorre el cielo y, cuando cruza nuestra línea de visión, produce un patrón detectable de emisión de radio de banda ancha. Como los púlsares giran rápidamente, este patrón se repite periódicamente. Por tanto, la búsqueda de púlsares implica buscar señales de radio periódicas con grandes radiotelescopios.

Cada púlsar produce un patrón de emisión algo diferente, que varía levemente con cada rotación. Por lo tanto, una detección de señal potencial conocida como «candidata» se promedia a lo largo de muchas rotaciones del púlsar, según lo determinado por la duración de una observación. A falta de información adicional, cada candidato podría describir un púlsar real. Sin embargo, en la práctica, casi todas las detecciones son causadas por interferencias de radiofrecuencia (RFI) y ruido, lo que dificulta encontrar señales legítimas.

El fichero `pulsar_stars.csv` contiene datos acerca de una serie de púlsares reales y de ejemplos espurios producidos por RFI y ruido. Cada candidato se describe mediante ocho atributos continuos extraídos de las señales recibidas.

Se pide realizar lo siguiente:

1. Dividir el conjunto de ejemplos en un subconjunto de entrenamiento (80&nbsp;% de los ejemplos) y un subconjunto de prueba (20&nbsp;% de los ejemplos). La división debe realizarse mediante muestreo estratificado, ya que la cantidad de ejemplos que se corresponden con púlsares reales es mucho menor que la de los que son interferencias y ruido.
2. Construir un árbol de decisión a partir del subconjunto de entrenamiento y calcular la matriz de confusión sobre el conjunto de prueba para cada combinación de los siguientes valores:
   - Máxima profundidad del árbol (argumento `max_depth`): de 1 a 5.
   - Cantidad mínima de ejemplos en las hojas (argumento `min_samples_leaf`): 1, 3 y 5.
   - Cantidad mínima de ejemplos para poder particionar (argumento `min_samples_split`): 10, 15 y 20.
3. De entre los árboles construidos en el apartado anterior seleccionar uno con máxima tasa de acierto sobre el conjunto de prueba, uno con máxima sensibilidad y uno con máxima precisión.

**Ayuda**: para los apartados 2 y 3 considerar el uso de [`ParameterGrid`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ParameterGrid.html) del módulo `model_selection`.

In [75]:
# Análisis de datos
import pandas as pd

pulsar_start = pd.read_csv('pulsar_stars.csv')

atributos = pulsar_start.loc[:,' Mean of the integrated profile':' Skewness of the DM-SNR curve']
objetivo = pulsar_start['target_class']

# 1. Dividir en entrenamiento y pruebas
from sklearn.model_selection import train_test_split
(atributos_entrenamiento, atributos_prueba, 
 objetivo_entrenamiento, objetivo_prueba) = train_test_split(atributos, objetivo, test_size=.2, stratify=objetivo)

# 2 Árbol de decisión
from sklearn.tree import DecisionTreeClassifier
from inspect import signature
print(signature(DecisionTreeClassifier))

rejilla_de_hiperparametros={'max_depth': range(1,6),
                            'min_samples_leaf': [1,3,5],
                            'min_samples_split': [10,15,20]
                            } 
# No puedo usar GridSearchCV porque no tengo un modelo específico, necesito iterar sobre todos los modelos posibles
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score
resultados = []
for parametros in ParameterGrid(rejilla_de_hiperparametros):
    modelo = DecisionTreeClassifier(**parametros)
    modelo.fit(atributos_entrenamiento, objetivo_entrenamiento)
    
    # Predicciones
    predicciones = modelo.predict(atributos_prueba)

    # Calcular métricas y guardar en resultados
    acc = accuracy_score(objetivo_prueba, predicciones)
    prec = precision_score(objetivo_prueba, predicciones)
    rec = recall_score(objetivo_prueba, predicciones)
    cm = confusion_matrix(objetivo_prueba, predicciones)
    resultados.append({
        'parametros': parametros,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'confusion_matrix': cm
    })
    
    
# ver resultados
resultados_df = pd.DataFrame(resultados)

resultados_df.sort_values(by='accuracy', ascending=False, inplace=True)


# 3. de entre todos los modelos, elegir el mejor sobre el conjunto de pruebas

bests_accuracy = resultados_df['accuracy'].max()
bests_models = resultados_df[resultados_df['accuracy'] == bests_accuracy]
print("--"*20)
print(f"Mejor modelo:\n{bests_models.iloc[0]}")

bests_recall = resultados_df['recall'].max()
bests_models_recall = resultados_df[resultados_df['recall'] == bests_recall]
print("--"*20)
print(f"Mejor modelo por recall:\n{bests_models_recall.iloc[0]}")

bests_precision = resultados_df['precision'].max()
bests_models_precision = resultados_df[resultados_df['precision'] == bests_precision]
print("--"*20)
print(f"Mejor modelo por precisión:\n{bests_models_precision.iloc[0]}")

#pulsar_start.head()

(*, criterion='gini', splitter='best', max_depth=None, min_samples_split=2, min_samples_leaf=1, min_weight_fraction_leaf=0.0, max_features=None, random_state=None, max_leaf_nodes=None, min_impurity_decrease=0.0, class_weight=None, ccp_alpha=0.0, monotonic_cst=None)
----------------------------------------
Mejor modelo:
parametros          {'max_depth': 5, 'min_samples_leaf': 1, 'min_s...
accuracy                                                     0.982123
precision                                                    0.931373
recall                                                       0.868902
confusion_matrix                              [[3231, 21], [43, 285]]
Name: 37, dtype: object
----------------------------------------
Mejor modelo por recall:
parametros          {'max_depth': 3, 'min_samples_leaf': 1, 'min_s...
accuracy                                                     0.978771
precision                                                    0.886503
recall                       

### Ejercicio 3

El hormigón es el material más importante en la ingeniería civil. La resistencia a la compresión del hormigón es una función altamente no lineal de su edad y sus ingredientes.

El fichero `concrete_data.csv` contiene la siguiente información acerca de diferentes muestras de hormigón:

* Contenido de cemento (`Cement`).
* Contenido de escoria de alto horno (`Blast Furnace Slag`).
* Contenido de cenizas volantes (`Fly Ash`).
* Contenido de agua (`Water`).
* Contenido de superplastificantes (`Superplasticizer`).
* Contenido de agregados gruesos (`Coarse Aggregate`).
* Contenido de agregados finos (`Fine Aggregate`).
* Edad del hormigón (`Age`).

El objetivo es predecir la resistencia a la compresión (`Strength`) a partir de esos atributos continuos.

Se pide realizar lo siguiente:

1. Definir una tubería que concatene un transformador de columnas que normalice los atributos al intervalo $[0, 1]$ y un modelo $k$NN para regresión.
2. Realizar una búsqueda en rejilla para estimar mediante validación cruzada el coeficiente de determinación obtenido al aplicar la tubería a cada combinación de los valores 1 a 5 para el número de vecinos y las distancias manhattan y euclídea para la métrica.
3. Repetir los pasos 1 y 2 usando ahora la tipificación (es decir, restar la media y dividir por la desviación típica) como procedimiento de normalización de los atributos.
4. Seleccionar el mejor procedimiento de normalización, el mejor valor para el número de vecinos y la mejor métrica y entrenar un modelo $k$NN a partir de todos los ejemplos usados para la validación cruzada, proporcionando finalmente su coeficiente de determinación sobre un conjunto de prueba reservado desde el principio.

**Ayuda**: las clases [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) y [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) del módulo `preprocessing` implementan los procedimientos de normalización, mientras que la clase [`KNeighborsRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) del módulo `neighbors` implementa el modelo $k$NN para una tarea de regresión.

### Ejercicio 4

Los [abulones](https://es.wikipedia.org/wiki/Haliotis) son una familia de moluscos gasterópodos. La edad de cada individuo está correlacionada con el número de anillos de su concha y, por tanto, puede determinarse cortando la concha a través del cono, tiñéndola y contando el número de anillos a través de un microscopio. Este procedimiento requiere mucho tiempo y es propenso a errores, por lo que sería preferible poder determinar la edad directamente a partir de medidas físicas más fáciles de obtener.

El fichero `abalone.csv` contiene la siguiente información de distintos individuos de abulones:

* Sexo (`Sex`): atributo discreto con posibles valores `M` (macho), `F` (hembra) e `I` (infante).
* Longitud (`Length`) en milímetros.
* Diámetro (`Diameter`) en milímetros.
* Altura (`Height`) en milímetros.
* Peso total (`Whole_weight`) en gramos.
* Peso sin la concha (`Shucked_weight`) en gramos.
* Peso intestinal (`Viscera_weight`) en gramos.
* Peso de la concha (`Shell_weight`) en gramos.

Se pide construir el mejor modelo posible para resolver la tarea de predecir el número de anillos (`Rings`) a partir de los atributos anteriores (entonces bastaría sumar 1.5 a ese número de anillos para obtener la edad, en años, del individuo).